In [ ]:
import pickle, os
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_core.documents import Document
from itertools import groupby
import re

CHUNKS_CACHE_PATH = "chunks_cache.pkl"

def normalize_arabic(text):
    if not text:
        return ""
    text = re.sub(r'[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED\u08D4-\u08FF]', '', text)
    text = text.replace("\u0640", "")
    text = re.sub(r'[\u0622\u0623\u0625\u0671]', '\u0627', text)
    text = re.sub(r'[\u0649\u06CC]', '\u064A', text)
    text = text.replace("\u0624", "\u0648")  # ؤ -> و
    text = text.replace("\u0626", "\u064A")  # ئ -> ي
    text = text.replace("\u0629", "\u0647")
    text = re.sub(r'[\u06DD\u06DE\u06E9\u08E2\u0660-\u0669\u06F0-\u06F9]', '', text)
    text = re.sub(r'[\ufd3e\ufd3f]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def chunk_quran_docs(docs, window_size=5, overlap=1):
    docs = sorted(docs, key=lambda d: (int(d.metadata["surah_no"]), int(d.metadata["ayah_no"])))
    chunks = []
    step = window_size - overlap

    for surah_no, group_iter in groupby(docs, key=lambda d: d.metadata["surah_no"]):
        group = list(group_iter)
        for start in range(0, len(group), step):
            window = group[start:start + window_size]
            if not window:
                break
            first, last = window[0], window[-1]
            text = " ".join(d.page_content for d in window)
            f_meta, l_meta = first.metadata, last.metadata
            ref = (f"{surah_no}:{f_meta['ayah_no']}"
                   if f_meta["ayah_no"] == l_meta["ayah_no"]
                   else f"{surah_no}:{f_meta['ayah_no']}-{l_meta['ayah_no']}")
            metadata = {
                "chunk_id": f"{int(surah_no):03d}_{int(f_meta['ayah_no']):03d}_{int(l_meta['ayah_no']):03d}",
                "surah_no": int(surah_no),
                "surah_name_ar": f_meta["surah_name_ar"],
                "surah_name_en": f_meta["surah_name_en"],
                "revelation_place": f_meta["revelation_place"],
                "start_ayah": int(f_meta["ayah_no"]),
                "end_ayah": int(l_meta["ayah_no"]),
                "reference": ref,
            }
            chunks.append(Document(page_content=text, metadata=metadata))
            if start + window_size >= len(group):
                break
    return chunks


if os.path.exists(CHUNKS_CACHE_PATH):
    # Fast path: skip CSV load + normalization + chunking entirely
    with open(CHUNKS_CACHE_PATH, "rb") as f:
        chunks = pickle.load(f)
    print(f"Loaded {len(chunks)} chunks from cache: {CHUNKS_CACHE_PATH}")
else:
    loader = CSVLoader(
        file_path="Quran_RAG_Ready.csv",
        encoding="utf-8-sig",
        content_columns=["text_ar"],
        metadata_columns=["chunk_id", "surah_no", "surah_name_ar",
                           "surah_name_en", "revelation_place",
                           "ayah_no", "ayah_no_global", "reference"],
    )
    csv_docs = loader.load()
    print(f"Number of documents loaded: {len(csv_docs)}")

    for doc in csv_docs:
        doc.page_content = normalize_arabic(doc.page_content)

    chunks = chunk_quran_docs(csv_docs, window_size=5, overlap=1)

    with open(CHUNKS_CACHE_PATH, "wb") as f:
        pickle.dump(chunks, f)
    print(f"Total chunks: {len(chunks)} (cached to {CHUNKS_CACHE_PATH})")

print(chunks[0].page_content)
print(chunks[0].metadata)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name ="BAAI/bge-m3",
    model_kwargs={"device": "cpu"}, 
    encode_kwargs={"normalize_embeddings": True},
)

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever #sparse
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document

texts = [doc.page_content for doc in chunks]
metadatas = [doc.metadata for doc in chunks]

vectorstore = FAISS.load_local(
    ".",
    embeddings=embedding_model,
    allow_dangerous_deserialization=True
)

print("FAISS loaded:", vectorstore is not None)
print("Index total vectors:", getattr(vectorstore.index, "ntotal", None))
print("Index dimension:", getattr(vectorstore.index, "d", None))
print("Has retriever:", hasattr(vectorstore, "as_retriever"))

In [ ]:
import pickle, os

BM25_CACHE_PATH = "sparse_retriever_cache.pkl"
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 8})  
                                           

if os.path.exists(BM25_CACHE_PATH):
    with open(BM25_CACHE_PATH, "rb") as f:
        sparse_retriever = pickle.load(f)
    print(f"Loaded BM25 retriever from cache: {BM25_CACHE_PATH}")
else:
    sparse_retriever = BM25Retriever.from_documents(chunks)
    with open(BM25_CACHE_PATH, "wb") as f:
        pickle.dump(sparse_retriever, f)
    print(f"Built BM25 retriever and cached to {BM25_CACHE_PATH}")

sparse_retriever.k = 8  

In [ ]:
hybrid_retriever= EnsembleRetriever(
retrievers=[dense_retriever,sparse_retriever],
weights=[0.4, 0.6])
hybrid_retriever

In [ ]:
query = "سورة كان سيدنا موسى  يأمر من قومه يذبحو بقرة بوحي من الله"
normalized_query = normalize_arabic(query)
results = hybrid_retriever.invoke(normalized_query)

# Step 6: Print results
for i, doc in enumerate(results):
    print(f"\n\U0001F539 Document {i+1}:\n{doc.page_content}")

In [49]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter,
    TokenTextSplitter,
)
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser 

In [50]:
from langchain_core.prompts import PromptTemplate

reranker_prompt = PromptTemplate.from_template("""
You are an expert retrieval reranker.

Your job is NOT to answer the question.

Your job is to rank the retrieved documents according to how useful they are for answering the user's question.

Question:
{question}

Retrieved Documents:
{documents}

Instructions:
1. Rank the documents from MOST relevant to LEAST relevant.
2. Prefer documents that directly answer the user's question.
3. Prefer exact matches over loosely related topics.
4. If multiple documents are related, rank the one containing the most complete answer first.
5. Ignore duplicate information.
6. Do NOT explain your reasoning.
7. Return ONLY the document indices.

Output format:
0,3,1,2,4
""")

In [51]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.1-8b-instant")

In [ ]:
# Separate model for reranking -- keeps this off the same TPM bucket as
# the main agent `llm`, since a single query can otherwise stack 2-3 calls
# (tool routing, rerank, final answer) against one model within seconds.
rerank_llm = ChatGroq(model="openai/gpt-oss-20b").with_retry(
    wait_exponential_jitter=True,
    stop_after_attempt=5,
)

chain = reranker_prompt | rerank_llm | StrOutputParser()

In [ ]:
def dedupe_overlapping_chunks(docs, overlap_threshold=0.5):
    """Drop chunks that heavily overlap (same surah, overlapping ayah range)
    with a chunk already kept, since window_size/overlap chunking produces
    near-duplicate neighboring windows that dilute the reranker's view."""
    selected = []
    for doc in docs:
        s = doc.metadata.get("surah_no")
        a = doc.metadata.get("start_ayah")
        b = doc.metadata.get("end_ayah")
        is_dup = False
        for kept in selected:
            s2 = kept.metadata.get("surah_no")
            a2 = kept.metadata.get("start_ayah")
            b2 = kept.metadata.get("end_ayah")
            if s == s2 and a is not None and b is not None and a2 is not None and b2 is not None:
                overlap = max(0, min(b, b2) - max(a, a2) + 1)
                span = min(b - a + 1, b2 - a2 + 1)
                if span > 0 and overlap / span >= overlap_threshold:
                    is_dup = True
                    break
        if not is_dup:
            selected.append(doc)
    return selected


deduped_results = dedupe_overlapping_chunks(results)

doc_lines = [f"{i}. [{doc.metadata.get('reference', '')}] {doc.page_content[:200]}"
             for i, doc in enumerate(deduped_results)]

ranked_output = chain.invoke({
    "question": query,
    "documents": formatted_docs
})

print("Reranked order (document indices):")
print(ranked_output)

order = [int(i.strip()) for i in ranked_output.split(",") if i.strip().isdigit()]
reranked_results = [deduped_results[i] for i in order if i < len(deduped_results)]

for i, doc in enumerate(reranked_results):
    ref = doc.metadata.get("reference", "")
    print(f"\n\U0001F538 Reranked {i+1} [{ref}]:\n{doc.page_content}")

In [ ]:
from langchain_core.prompts import PromptTemplate

answer_prompt = PromptTemplate.from_template("""
You are a knowledgeable assistant answering questions about the Quran using only the verses provided as context.

Question:
{question}

Relevant verses (each prefixed with its surah:ayah reference):
{context}

Instructions:
1. Answer using only the information in the verses above — do not use outside knowledge.
2. If the verses don't contain enough information to answer, say so honestly instead of guessing.
3. Cite the surah:ayah reference(s) shown in brackets for every claim you make — do not invent references not present in context.
4. Keep the answer concise and respectful of the religious nature of the text.

Answer:
""")

answer_chain = answer_prompt | llm | StrOutputParser()

context_docs = reranked_results if reranked_results else results
final_context = "\n".join(
    f"[{doc.metadata.get('reference', '')}] {doc.page_content}"
    for doc in context_docs[:5]
)

final_answer = answer_chain.invoke({
    "question": query,
    "context": final_context
})

DISCLAIMER = (
    "\n\n---\n"
    "This is an AI-assisted retrieval tool, not a scholarly ruling. "
    "For religious guidance, please consult a qualified scholar."
)

print("Final Answer:\n")
print(final_answer + DISCLAIMER)

### AGENT

In [ ]:
from langchain_community.tools import (
    ArxivQueryRun,
    WikipediaQueryRun,
    GoogleSerperRun,
    GoogleSerperResults,
)
from langchain_community.utilities import WikipediaAPIWrapper, ArxivAPIWrapper
import wikipedia
import arxiv as arxiv_pkg

wikipedia.set_user_agent("AgenticAI/1.0 (https://example.com)")

def _fetch_results_compat(self, query: str):
    if self.is_arxiv_identifier(query):
        search = self.arxiv_search(
            id_list=query.split(), max_results=self.top_k_results
        )
    else:
        search = self.arxiv_search(
            query[: self.ARXIV_MAX_QUERY_LENGTH], max_results=self.top_k_results
        )
    return list(arxiv_pkg.Client().results(search))

ArxivAPIWrapper._fetch_results = _fetch_results_compat

arxiv_api_wrapper = ArxivAPIWrapper(top_k_results=2,doc_content_chars_max=500)
arxiv = ArxivQueryRun(api_wrapper=arxiv_api_wrapper)
arxiv.name

In [ ]:
results = arxiv.run("attention is all you need")

In [ ]:
### Wikipedia Tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

wiki_api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=500)
wiki = WikipediaQueryRun(api_wrapper=wiki_api_wrapper)
wiki.name

In [ ]:
### Tavily Search Tool
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

from langchain_community.tools.tavily_search import TavilySearchResults

tavily = TavilySearchResults()
tavily.name

In [ ]:
### Quran RAG Tool
from langchain_core.tools import tool

@tool
def quran_search(query: str) -> str:
    """Search the Quran for verses relevant to a question or topic.
    Use this whenever the user asks about Quranic verses, Islamic teachings,
    stories from the Quran (e.g. Musa, Ibrahim), or wants a Quran-grounded answer.
    Returns the most relevant verses along with their surah:ayah references.
    """
    normalized_query = normalize_arabic(query)
    raw_results = hybrid_retriever.invoke(normalized_query)
    deduped = dedupe_overlapping_chunks(raw_results)

    # Limit number of documents and truncate content to avoid LLM token limits
    MAX_RERANK_DOCS = 20
    DOC_MAX_CHARS = 1000
    limited = deduped[:MAX_RERANK_DOCS]

    doc_lines = [f"{i}. [{doc.metadata.get('reference', '')}] {doc.page_content[:DOC_MAX_CHARS]}"
                 for i, doc in enumerate(limited)]
    formatted_docs = "
".join(doc_lines)

    ranked_output = chain.invoke({"question": query, "documents": formatted_docs})
    order = [int(i.strip()) for i in ranked_output.split(",") if i.strip().isdigit()]
    reranked = [limited[i] for i in order if i < len(limited)]

    top_docs = reranked[:5] if reranked else limited[:5]
    return "
".join(f"[{d.metadata.get('reference', '')}] {d.page_content[:DOC_MAX_CHARS]}" for d in top_docs)


In [ ]:
### Hadith API (fawazahmed0/hadith-api, free, no key required)
import requests, json, os

HADITH_CACHE_PATH = "hadith_bukhari_eng.json"
HADITH_EDITION = "eng-bukhari"  # English translation of Sahih al-Bukhari
HADITH_URL = f"https://cdn.jsdelivr.net/gh/fawazahmed0/hadith-api@1/editions/{HADITH_EDITION}.json"

def load_hadith_collection():
    if os.path.exists(HADITH_CACHE_PATH):
        with open(HADITH_CACHE_PATH, "r", encoding="utf-8") as f:
            return json.load(f)
    resp = requests.get(HADITH_URL, timeout=30)
    resp.raise_for_status()
    data = resp.json()
    with open(HADITH_CACHE_PATH, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False)
    return data

hadith_collection = load_hadith_collection()
hadith_list = hadith_collection.get("hadiths", [])
print(f"Loaded {len(hadith_list)} hadiths from {HADITH_EDITION}")
print("Example record keys:", list(hadith_list[0].keys()) if hadith_list else None)

In [ ]:
### Hadith Search Tool
@tool
def hadith_search(query: str, max_results: int = 5) -> str:
    """Search Sahih al-Bukhari hadiths (English translation) for a keyword or topic.
    Use this when the user asks about a hadith, or a saying/action of the Prophet
    Muhammad (peace be upon him), or wants a hadith to complement a Quranic answer.
    Returns matching hadiths with their hadith numbers.
    """
    query_lower = query.lower()
    matches = [h for h in hadith_list if query_lower in h.get("text", "").lower()]

    if not matches:
        # fall back to matching on individual keywords if the exact phrase isn't found
        words = [w for w in query_lower.split() if len(w) > 3]
        matches = [h for h in hadith_list if any(w in h.get("text", "").lower() for w in words)]

    top = matches[:max_results]
    if not top:
        return "No matching hadiths found."

    return "\n\n".join(
        f"Hadith #{h.get('hadithnumber')}: {h.get('text')}" for h in top
    )

In [ ]:
### DuckDuckGo Search Tool (free, no API key)
# pip install -U duckduckgo-search
from langchain_community.tools import DuckDuckGoSearchRun

ddg_search = DuckDuckGoSearchRun()
ddg_search.name

In [ ]:
### Combine all tools and bind to the LLM
tools = [arxiv, wiki, tavily, ddg_search, quran_search, hadith_search]

llm_with_tools = llm.bind_tools(tools).with_retry(
    wait_exponential_jitter=True,
    stop_after_attempt=5,
)

In [ ]:
from langchain_core.messages import SystemMessage

system_prompt = SystemMessage(content="""You are an assistant that answers questions about the Quran and Islamic teachings.
Always prefer the quran_search tool for any question about Quranic verses, stories, or teachings — it returns grounded, citable verses.
Use hadith_search when the user asks about a hadith or a saying/action of the Prophet Muhammad (peace be upon him), or to complement a Quran answer with a related hadith.
Use wikipedia for general background/historical context.
Use arxiv for questions about research papers.
Use tavily_search or ddg_search only for current events or information not related to religious text — prefer tavily_search first, and use ddg_search as a fallback if tavily is unavailable.
Never answer Quran-related or hadith-related questions from your own knowledge without calling quran_search or hadith_search first.
""")

In [ ]:
## State Schema
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage, HumanMessage
from typing import Annotated
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

In [ ]:
### Entire Chatbot With LangGraph
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

def tool_calling_llm(state: State):
    messages = state["messages"]
    if not any(isinstance(m, SystemMessage) for m in messages):
        messages = [system_prompt] + messages
    return {"messages": [llm_with_tools.invoke(messages)]}

# Build graph
builder = StateGraph(State)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "tool_calling_llm")
builder.add_conditional_edges(
    "tool_calling_llm",
    # If the latest message from the assistant is a tool call -> route to tools
    # Otherwise -> route to END
    tools_condition,
)
builder.add_edge("tools", "tool_calling_llm")

graph = builder.compile()

# View
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
messages = graph.invoke({"messages": HumanMessage(content="What does the Quran say about patience during hardship?")})
for m in messages["messages"]:
    m.pretty_print()

In [ ]:
### Agent Memory (optional) -- lets the agent remember prior turns in a conversation
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
graph_memory = builder.compile(checkpointer=memory)

config = {"configurable": {"thread_id": "1"}}

messages = graph_memory.invoke(
    {"messages": [HumanMessage(content="What does the Quran say about Musa and Pharaoh?")]},
    config=config
)
for m in messages["messages"]:
    m.pretty_print()